# Act 2: Cruise Watch Telemetry Anomaly Detection

Spacecraft telemetry streams experience sudden equipment faults like stuck sensors, voltage drifts, and noise bursts. We extract sliding window statistics across multiple sensors and train Isolation Forests alongside neural Autoencoders to learn normal operational behavior. When observed signals deviate strongly from the learned baseline, the detector triggers an anomaly alarm. Tuning window size and detection persistence prevents spurious false alarms while catching dangerous faults quickly.

In [ ]:
# Setup and package installation
REPO_URL = "https://github.com/placeholder/meridian"
# Option 1: Install from GitHub in Google Colab
# !pip install -q git+$REPO_URL

# Option 2: Install from local editable package
# !pip install -e /path/to/meridian-workshop

import sys
from pathlib import Path
repo_dir = Path.cwd()
while repo_dir.parent != repo_dir and not (repo_dir / "meridian").exists():
    repo_dir = repo_dir.parent
if (repo_dir / "meridian").exists() and str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))


In [ ]:
import os
TEAM_ID = os.environ.get("MERIDIAN_TEAM_ID", "your-team-name")

from meridian.state import load_state
from meridian.ui import briefing, dashboard, debrief
state = load_state(TEAM_ID)


In [ ]:
briefing(2)

### Generating Cruise Telemetry Stream

The length of the cruise stream depends directly on the orbital period selected in Act 1. Four vital probe subsystems are monitored: bus voltage, battery temperature, reaction wheel speed, and solar current. We simulate unexpected hardware faults injected at unknown intervals along the flight path.

In [ ]:
from meridian.act2_telemetry import make_stream, train, run, apply_to_state, plot_result, human_vs_ai
trip_days = state.act1.target.trip_days if state.act1.done else 100
stream = make_stream(TEAM_ID, trip_days=trip_days)
print(f'Cruise telemetry stream generated: {len(stream.mission_data["bus_voltage"])} steps, {len(stream.faults)} injected faults.')

### Human vs AI Warm-up

Can you visually spot subtle equipment malfunctions within high-frequency sensor readings? Examine three segments from the probe telemetry stream below. One segment contains a genuine sensor fault while the other two display normal operating noise.

In [ ]:
# Spot the faulty sensor stream segment
h_res = human_vs_ai(stream)

### YOUR TURN: Configure Anomaly Detection

Select your anomaly detection model ('iforest', 'autoencoder', or 'both') and adjust window size and threshold. Higher threshold percentiles reduce false alarms but may delay catching slow drifts. Persistence requires consecutive anomalous windows before firing an alarm to filter out isolated noise spikes.

In [ ]:
# YOUR TURN: Adjust Act 2 Anomaly Detection hyperparameters
knobs = {
    "detector": "both",
    "window": 64,
    "threshold_q": 99.0,
    "persistence": 3,
}

In [ ]:
# Train detector and evaluate autonomous anomaly detection on mission telemetry
detector = train(stream, **knobs)
result = run(detector, stream)
apply_to_state(state, result)
print(f'Act 2 Score: {result.score:.1f}/200 | Caught: {result.caught}/{result.n_faults} | False Alarms: {result.false_alarms}')

In [ ]:
# Plot detected alarms and actual faults across all 4 channels
import matplotlib.pyplot as plt
plot_result(stream, result)
plt.show()

In [ ]:
debrief(state, 2)

In [ ]:
state.save()
dashboard(state)